# Clustering - Real Dataset Applications
### Part 1: K-Means on Iris Dataset | Part 2: Hierarchical on Wine Dataset | Part 3: DBSCAN on Mall Customers | Capstone

**Datasets Used:**
- **Iris** (sklearn) — 150 flower measurements, 3 species (use as unsupervised)
- **Wine** (sklearn) — 178 wine samples, 13 chemical features, 3 classes
- **Mall Customers** — synthetic but realistic customer dataset
- **Capstone** — Credit card fraud detection (DBSCAN for anomaly detection)

## Imports

**New Keywords (review):**
- `load_iris`, `load_wine` — sklearn built-in real datasets
- `PCA` — Principal Component Analysis (reduces dimensions for visualization)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
print("Libraries loaded!")

---
## Part 1: K-Means Clustering on Iris Dataset

### About the Iris Dataset:
- 150 flowers from 3 species: setosa, versicolor, virginica
- 4 features: sepal length, sepal width, petal length, petal width
- We will **pretend we don't know the species** and let K-Means find them!

**New Keyword:**
- `adjusted_rand_score` — compares predicted clusters to true labels (0=random, 1=perfect)

In [ ]:
# Load Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df_iris = pd.DataFrame(X_iris, columns=feature_names)
df_iris['true_species'] = [target_names[y] for y in y_iris]

print("Dataset shape:", df_iris.shape)
print("Species distribution:\n", df_iris['true_species'].value_counts())
print(df_iris.head())

### EDA - Pairplot on Iris

**New Keyword:**
- `sns.pairplot` — plots every feature against every other feature. Great for spotting natural clusters!

In [ ]:
# Pairplot - visualize all feature combinations
sns.pairplot(df_iris, hue='true_species', diag_kind='kde', height=1.8)
plt.suptitle('Iris Dataset - Pairplot by True Species', y=1.02)
plt.tight_layout()
plt.show()
print("Notice: setosa is well-separated! versicolor and virginica overlap.")

### Elbow Method + Silhouette to Find K

In [ ]:
# Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Elbow + Silhouette
inertias, sil_scores = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbls = km.fit_predict(X_iris_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_iris_scaled, lbls))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-', markersize=8)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, sil_scores, 'gs-', markersize=8)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (Higher = Better)'); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Best K by silhouette: {K_range[sil_scores.index(max(sil_scores))]}")

In [ ]:
# Apply K-Means with K=3
km_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
iris_pred = km_iris.fit_predict(X_iris_scaled)

# How well did it match true labels?
ari = adjusted_rand_score(y_iris, iris_pred)
sil = silhouette_score(X_iris_scaled, iris_pred)
print(f"Adjusted Rand Index: {ari:.4f}  (1.0 = perfect match)")
print(f"Silhouette Score:    {sil:.4f}")

# Visualize with PCA (reduce 4D → 2D for plotting)
pca = PCA(n_components=2)
X_iris_2d = pca.fit_transform(X_iris_scaled)
print(f"\nVariance explained by 2 PCA components: {sum(pca.explained_variance_ratio_)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_iris_2d[:,0], X_iris_2d[:,1], c=y_iris, cmap='tab10', s=40, alpha=0.7)
axes[0].set_title('True Species (Ground Truth)')
axes[1].scatter(X_iris_2d[:,0], X_iris_2d[:,1], c=iris_pred, cmap='tab10', s=40, alpha=0.7)
axes[1].set_title(f'K-Means K=3 (ARI={ari:.3f})')
for ax in axes:
    ax.set_xlabel('PCA Component 1'); ax.set_ylabel('PCA Component 2')
plt.suptitle('Iris: True Labels vs K-Means Clusters (PCA Projection)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Part 2: K-Means Color Quantization on a Realistic Image

We will create a realistic multi-color test image and apply color quantization for different K values to compare compression quality.

**New Keywords:**
- `img.reshape(-1, 3)` — flatten image from (H, W, 3) to (pixels, 3)
- Comparing different K = different compression levels

In [ ]:
np.random.seed(42)
h, w = 150, 150

# Create a gradient + pattern image (more realistic than solid blocks)
x_coords = np.linspace(0, 1, w)
y_coords = np.linspace(0, 1, h)
xx, yy = np.meshgrid(x_coords, y_coords)

img_real = np.zeros((h, w, 3), dtype=np.uint8)
img_real[:, :, 0] = (np.sin(xx * np.pi * 3) * 127 + 128).astype(np.uint8)   # Red channel
img_real[:, :, 1] = (np.sin(yy * np.pi * 2) * 127 + 128).astype(np.uint8)   # Green channel
img_real[:, :, 2] = ((xx + yy) / 2 * 255).astype(np.uint8)                   # Blue channel

# Add some noise
noise = np.random.randint(-15, 15, img_real.shape)
img_real = np.clip(img_real.astype(int) + noise, 0, 255).astype(np.uint8)

print("Image shape:", img_real.shape)
print("Unique colors:", len(np.unique(img_real.reshape(-1,3), axis=0)))

plt.figure(figsize=(4, 4))
plt.imshow(img_real)
plt.title("Original Image")
plt.axis('off')
plt.show()

In [ ]:
# Compare different K values for color quantization
pixels = img_real.reshape(-1, 3).astype(float)

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].imshow(img_real); axes[0].set_title('Original'); axes[0].axis('off')

for i, k in enumerate([2, 4, 8, 16], 1):
    km_c = KMeans(n_clusters=k, random_state=42, n_init=5)
    km_c.fit(pixels)
    q_pixels = km_c.cluster_centers_[km_c.labels_]
    q_img = q_pixels.reshape(h, w, 3).astype(np.uint8)
    unique_colors = len(np.unique(q_img.reshape(-1,3), axis=0))
    axes[i].imshow(q_img)
    axes[i].set_title(f'K={k}\n({unique_colors} colors)')
    axes[i].axis('off')

plt.suptitle('Color Quantization: Fewer Colors = More Compression', fontsize=12)
plt.tight_layout()
plt.show()
print("K=2: very compressed (bad quality)")
print("K=16: close to original")

---
## Part 3: Hierarchical Clustering on Wine Dataset

### About the Wine Dataset:
- 178 wine samples from 3 Italian wineries
- 13 chemical features (alcohol, malic acid, ash, etc.)
- We use Hierarchical clustering and read K from the dendrogram

In [ ]:
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

df_wine = pd.DataFrame(X_wine, columns=wine.feature_names)
df_wine['true_class'] = y_wine

print("Wine Dataset Shape:", df_wine.shape)
print("Class distribution:\n", df_wine['true_class'].value_counts())
print(df_wine.describe().round(2).T)

In [ ]:
# Scale wine data (important - features have very different scales)
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# Dendrogram (use full 178 samples)
Z_wine = linkage(X_wine_scaled, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z_wine,
           leaf_rotation=90,
           leaf_font_size=5,
           color_threshold=10,
           truncate_mode='lastp',  # show only last p merges
           p=50)
plt.axhline(y=10, color='red', linestyle='--', label='Cut: K=3')
plt.title('Wine Dataset - Hierarchical Dendrogram (Truncated to 50 merges)')
plt.xlabel('Sample Index or Cluster Size (in brackets)')
plt.ylabel('Distance')
plt.legend()
plt.tight_layout()
plt.show()
print("3 main branches visible → K=3 is suggested by the dendrogram!")

In [ ]:
# Apply Hierarchical Clustering
hc_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
wine_pred = hc_wine.fit_predict(X_wine_scaled)

ari_wine = adjusted_rand_score(y_wine, wine_pred)
sil_wine = silhouette_score(X_wine_scaled, wine_pred)
print(f"Wine Hierarchical Clustering:")
print(f"  Adjusted Rand Index: {ari_wine:.4f}")
print(f"  Silhouette Score:    {sil_wine:.4f}")

# Visualize with PCA
pca_wine = PCA(n_components=2)
X_wine_2d = pca_wine.fit_transform(X_wine_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_wine_2d[:,0], X_wine_2d[:,1], c=y_wine, cmap='tab10', s=40, alpha=0.7)
axes[0].set_title('True Wine Classes')
axes[1].scatter(X_wine_2d[:,0], X_wine_2d[:,1], c=wine_pred, cmap='tab10', s=40, alpha=0.7)
axes[1].set_title(f'Hierarchical (ARI={ari_wine:.3f})')
for ax in axes:
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
plt.suptitle('Wine: True vs Hierarchical Clustering', fontsize=12)
plt.tight_layout(); plt.show()

---
## Part 4: DBSCAN on Mall Customers Dataset

### About the Dataset:
- 200 mall customers
- Features: annual_income, spending_score
- Some customers have unusual spending patterns (outliers)
- DBSCAN will find the natural groups AND flag the unusual customers

In [ ]:
np.random.seed(42)
n_mall = 200

# 5 customer groups + some outliers
groups = [
    {'income': (15000, 40000), 'score': (1, 40), 'n': 40},    # low income, low spend
    {'income': (15000, 40000), 'score': (60, 100), 'n': 40},   # low income, high spend
    {'income': (60000, 90000), 'score': (40, 60), 'n': 40},    # mid income, mid spend
    {'income': (90000, 140000), 'score': (1, 40), 'n': 40},    # high income, low spend
    {'income': (90000, 140000), 'score': (60, 100), 'n': 40},  # high income, high spend
]

records = []
for g in groups:
    inc = np.random.uniform(g['income'][0], g['income'][1], g['n'])
    sco = np.random.uniform(g['score'][0], g['score'][1], g['n'])
    records.extend(zip(inc, sco))

# Add a few outliers
outlier_income = [30000, 160000, 80000, 50000]
outlier_score = [95, 95, 1, 50]
records.extend(zip(outlier_income, outlier_score))

df_mall = pd.DataFrame(records, columns=['annual_income', 'spending_score'])
print("Mall Dataset Shape:", df_mall.shape)
print(df_mall.describe().round(1))

In [ ]:
# k-distance graph to find eps
X_mall = StandardScaler().fit_transform(df_mall)
nbrs = NearestNeighbors(n_neighbors=4).fit(X_mall)
dists, _ = nbrs.kneighbors(X_mall)
k_dists = np.sort(dists[:, -1])[::-1]

plt.figure(figsize=(8, 3))
plt.plot(k_dists, 'b-')
plt.axhline(y=0.5, color='red', linestyle='--', label='Optimal eps ≈ 0.5')
plt.title('K-Distance Graph - Mall Customers')
plt.xlabel('Points'); plt.ylabel('4-NN Distance')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Apply DBSCAN
db_mall = DBSCAN(eps=0.5, min_samples=5)
mall_labels = db_mall.fit_predict(X_mall)

n_clusters = len(set(mall_labels)) - (1 if -1 in mall_labels else 0)
n_noise = (mall_labels == -1).sum()
print(f"Clusters found: {n_clusters}")
print(f"Noise/Outlier points: {n_noise}")

# Visualize
plt.figure(figsize=(9, 6))
mask_normal = mall_labels != -1
mask_noise = mall_labels == -1

scatter = plt.scatter(
    df_mall.loc[mask_normal, 'annual_income'],
    df_mall.loc[mask_normal, 'spending_score'],
    c=mall_labels[mask_normal], cmap='tab10', s=50, alpha=0.7, label='Cluster Points'
)
plt.scatter(
    df_mall.loc[mask_noise, 'annual_income'],
    df_mall.loc[mask_noise, 'spending_score'],
    c='red', marker='X', s=150, zorder=5, label=f'Anomalies (n={n_noise})'
)
plt.xlabel('Annual Income'); plt.ylabel('Spending Score')
plt.title(f'DBSCAN: {n_clusters} Customer Segments + {n_noise} Anomalies')
plt.legend(); plt.tight_layout(); plt.show()

---
## Capstone Project: Credit Card Transaction Anomaly Detection

**Business Problem:** A bank wants to automatically detect unusual (potentially fraudulent) credit card transactions.

**Strategy:** Use DBSCAN on transaction features. Normal transactions form dense clusters. Unusual/fraudulent transactions = isolated noise points (-1).

**Features:**
- `amount` — transaction amount
- `hour_of_day` — hour when transaction happened (0-23)
- `merchant_category_code` — type of merchant (encoded)
- `distance_from_home` — how far from home city

In [ ]:
np.random.seed(42)
n_normal = 500
n_fraud = 25

# Normal transactions (3 clusters: small daily, medium shopping, large bill pay)
cluster1 = np.column_stack([
    np.random.normal(50, 20, 200),         # small amounts
    np.random.choice(range(8, 20), 200),   # daytime hours
    np.random.choice([1,2,3], 200),        # grocery/food/gas
    np.random.normal(2, 1, 200)            # near home
])
cluster2 = np.column_stack([
    np.random.normal(200, 50, 150),
    np.random.choice(range(10, 22), 150),
    np.random.choice([4,5,6], 150),
    np.random.normal(10, 3, 150)
])
cluster3 = np.column_stack([
    np.random.normal(800, 100, 150),
    np.random.choice(range(9, 18), 150),
    np.random.choice([7,8], 150),
    np.random.normal(5, 2, 150)
])

# Fraudulent transactions (unusual amounts + odd hours + far from home)
fraud = np.column_stack([
    np.random.uniform(2000, 5000, n_fraud),   # very high amounts
    np.random.choice(range(0, 5), n_fraud),   # 2-4 AM
    np.random.choice([9,10], n_fraud),        # unusual merchants
    np.random.normal(500, 100, n_fraud)       # very far from home
])

X_txn = np.vstack([cluster1, cluster2, cluster3, fraud])
is_fraud_true = np.array([0]*n_normal + [1]*n_fraud)
df_txn = pd.DataFrame(X_txn, columns=['amount', 'hour', 'merchant_code', 'distance_home'])
print("Transaction Dataset Shape:", df_txn.shape)
print("True frauds:", is_fraud_true.sum())

In [ ]:
# Scale and find eps
X_txn_scaled = StandardScaler().fit_transform(X_txn)
nbrs = NearestNeighbors(n_neighbors=4).fit(X_txn_scaled)
dists, _ = nbrs.kneighbors(X_txn_scaled)
k_dists = np.sort(dists[:,-1])[::-1]

plt.figure(figsize=(8, 3))
plt.plot(k_dists, 'b-')
plt.axhline(y=0.7, color='red', linestyle='--', label='eps ≈ 0.7')
plt.title('K-Distance Graph - Transactions')
plt.xlabel('Transactions (sorted)'); plt.ylabel('4-NN Distance')
plt.legend(); plt.tight_layout(); plt.show()

# Apply DBSCAN
db_fraud = DBSCAN(eps=0.7, min_samples=8)
fraud_pred_labels = db_fraud.fit_predict(X_txn_scaled)

detected_anomalies = (fraud_pred_labels == -1)
print(f"\nTotal transactions: {len(X_txn)}")
print(f"Flagged as anomaly (noise): {detected_anomalies.sum()}")
print(f"True frauds: {is_fraud_true.sum()}")

# Check detection
true_positives = ((detected_anomalies) & (is_fraud_true == 1)).sum()
false_positives = ((detected_anomalies) & (is_fraud_true == 0)).sum()
print(f"\nTrue Frauds Caught: {true_positives}/{n_fraud}")
print(f"False Alarms:        {false_positives}")

In [ ]:
# Visualize: Amount vs Distance from Home (2D view)
plt.figure(figsize=(10, 6))

# Normal transactions
mask_normal = fraud_pred_labels != -1
plt.scatter(df_txn.loc[mask_normal, 'amount'],
            df_txn.loc[mask_normal, 'distance_home'],
            c=fraud_pred_labels[mask_normal], cmap='tab10',
            s=20, alpha=0.5, label='Normal Transactions')

# Flagged anomalies
mask_flagged = fraud_pred_labels == -1
plt.scatter(df_txn.loc[mask_flagged, 'amount'],
            df_txn.loc[mask_flagged, 'distance_home'],
            c='red', marker='X', s=100, zorder=5,
            label=f'Flagged Anomalies (n={mask_flagged.sum()})')

plt.xlabel('Transaction Amount')
plt.ylabel('Distance from Home')
plt.title('DBSCAN Credit Card Fraud Detection')
plt.legend()
plt.tight_layout()
plt.show()
print("Red X = transactions flagged as unusual by DBSCAN")

## Final Summary

### What You Learned in Notebook 2:

| Algorithm | Dataset Used | Key Metric | Result |
|---|---|---|---|
| K-Means | Iris (150 samples, 4 features) | ARI ≈ 0.73 | Good match to true species |
| K-Means Color Quantization | Gradient Image (150×150) | Visual quality | K=16 close to original |
| Hierarchical | Wine (178 samples, 13 features) | ARI ≈ 0.90 | Excellent match! |
| DBSCAN | Mall Customers (200 samples) | Noise detection | Found outliers automatically |
| DBSCAN | Credit Card Transactions (525) | Fraud detection | Flagged unusual transactions |

### The Golden Rules:
1. **Always scale** features before clustering (StandardScaler)
2. **Use Elbow + Silhouette** together to find optimal K
3. **Use Dendrogram** to visualize and decide K for Hierarchical
4. **Use K-Distance Graph** to find optimal eps for DBSCAN
5. **Visualize** every result — clustering is exploratory!